In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🏠 Indian House Price Prediction - Exploratory Data Analysis\n",
    "## ML Internship Task 01 - Linear Regression Model\n",
    "\n",
    "### **Objective:**\n",
    "To analyze and understand the Indian housing dataset before building a linear regression model to predict house prices based on square footage, bedrooms, and bathrooms."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Import Required Libraries"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Data manipulation\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "\n",
    "# Data visualization\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "# Statistical analysis\n",
    "from scipy import stats\n",
    "from scipy.stats import norm\n",
    "\n",
    "# Warnings\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Set style for better visualizations\n",
    "plt.style.use('seaborn-v0_8-darkgrid')\n",
    "sns.set_palette(\"husl\")\n",
    "sns.set_style('whitegrid')\n",
    "\n",
    "# Display settings\n",
    "pd.set_option('display.max_columns', None)\n",
    "pd.set_option('display.max_rows', 100)\n",
    "pd.set_option('display.float_format', '{:.2f}'.format)\n",
    "\n",
    "print(\"✅ Libraries imported successfully!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Load the Dataset"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load the dataset - UPDATE THIS PATH TO YOUR ACTUAL FILE\n",
    "file_path = '../data/dataset.csv'  # Change this to your actual filename\n",
    "\n",
    "try:\n",
    "    df = pd.read_csv(file_path)\n",
    "    print(\"✅ Dataset loaded successfully!\")\n",
    "except FileNotFoundError:\n",
    "    print(\"❌ File not found. Please check the file path.\")\n",
    "    # Alternative: Try to find any CSV file in the data directory\n",
    "    import os\n",
    "    data_dir = '../data/'\n",
    "    csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]\n",
    "    if csv_files:\n",
    "        print(f\"\\nFound CSV files: {csv_files}\")\n",
    "        file_path = os.path.join(data_dir, csv_files[0])\n",
    "        df = pd.read_csv(file_path)\n",
    "        print(f\"✅ Loaded: {csv_files[0]}\")\n",
    "    else:\n",
    "        print(\"❌ No CSV files found in data directory\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Initial Data Overview"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"=\"*80)\n",
    "print(\"DATASET OVERVIEW\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "# Basic information\n",
    "print(f\"\\n📊 Dataset Shape: {df.shape}\")\n",
    "print(f\"   - Number of rows: {df.shape[0]}\")\n",
    "print(f\"   - Number of columns: {df.shape[1]}\")\n",
    "\n",
    "# First few rows\n",
    "print(\"\\n👀 First 5 rows of the dataset:\")\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Last few rows\n",
    "print(\"👀 Last 5 rows of the dataset:\")\n",
    "df.tail()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Column information\n",
    "print(\"\\n📋 Column Names and Data Types:\")\n",
    "df_info = pd.DataFrame({\n",
    "    'Column': df.columns,\n",
    "    'Data Type': df.dtypes.values,\n",
    "    'Non-Null Count': df.count().values,\n",
    "    'Null Count': df.isnull().sum().values,\n",
    "    'Null %': (df.isnull().sum() / len(df) * 100).values\n",
    "})\n",
    "df_info"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Statistical summary\n",
    "print(\"\\n📈 Statistical Summary:\")\n",
    "df.describe()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Identify Target and Feature Columns"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Function to identify potential columns\n",
    "def identify_columns(df):\n",
    "    \"\"\"Identify potential price and feature columns\"\"\"\n",
    "    \n",
    "    # Keywords for different types of columns\n",
    "    price_keywords = ['price', 'cost', 'value', 'amount', 'rent', 'sale', 'lakh', 'crore', 'rate']\n",
    "    bedroom_keywords = ['bed', 'bhk', 'room', 'bedroom', 'bed rooms']\n",
    "    bathroom_keywords = ['bath', 'washroom', 'bathroom', 'toilet', 'bath rooms']\n",
    "    area_keywords = ['area', 'sqft', 'size', 'sq_ft', 'square', 'built_up', 'carpet', 'super', 'plot']\n",
    "    \n",
    "    results = {\n",
    "        'price_columns': [],\n",
    "        'bedroom_columns': [],\n",
    "        'bathroom_columns': [],\n",
    "        'area_columns': []\n",
    "    }\n",
    "    \n",
    "    for col in df.columns:\n",
    "        col_lower = col.lower()\n",
    "        \n",
    "        # Check price columns\n",
    "        if any(keyword in col_lower for keyword in price_keywords):\n",
    "            results['price_columns'].append(col)\n",
    "        \n",
    "        # Check bedroom columns\n",
    "        if any(keyword in col_lower for keyword in bedroom_keywords):\n",
    "            results['bedroom_columns'].append(col)\n",
    "        \n",
    "        # Check bathroom columns\n",
    "        if any(keyword in col_lower for keyword in bathroom_keywords):\n",
    "            results['bathroom_columns'].append(col)\n",
    "        \n",
    "        # Check area columns\n",
    "        if any(keyword in col_lower for keyword in area_keywords):\n",
    "            results['area_columns'].append(col)\n",
    "    \n",
    "    return results\n",
    "\n",
    "# Identify columns\n",
    "identified = identify_columns(df)\n",
    "\n",
    "print(\"=\"*80)\n",
    "print(\"IDENTIFIED COLUMNS\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "print(\"\\n💰 Potential Price Columns:\")\n",
    "for i, col in enumerate(identified['price_columns'], 1):\n",
    "    print(f\"   {i}. {col} - Sample: {df[col].iloc[0]}\")\n",
    "\n",
    "print(\"\\n🛏️ Potential Bedroom Columns:\")\n",
    "for i, col in enumerate(identified['bedroom_columns'], 1):\n",
    "    print(f\"   {i}. {col} - Sample: {df[col].iloc[0]}\")\n",
    "\n",
    "print(\"\\n🚿 Potential Bathroom Columns:\")\n",
    "for i, col in enumerate(identified['bathroom_columns'], 1):\n",
    "    print(f\"   {i}. {col} - Sample: {df[col].iloc[0]}\")\n",
    "\n",
    "print(\"\\n📐 Potential Area Columns:\")\n",
    "for i, col in enumerate(identified['area_columns'], 1):\n",
    "    print(f\"   {i}. {col} - Sample: {df[col].iloc[0]}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Data Cleaning and Preprocessing"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create a cleaned dataframe\n",
    "df_clean = df.copy()\n",
    "\n",
    "# IMPORTANT: Update these column names based on your dataset\n",
    "# Based on the identification above, set the correct column names\n",
    "price_col = 'price'  # Change this to your actual price column\n",
    "bedroom_col = 'bedRoom'  # Change this to your actual bedroom column\n",
    "bathroom_col = 'bathroom'  # Change this to your actual bathroom column\n",
    "area_col = 'area'  # Change this to your actual area column\n",
    "\n",
    "print(\"⚠️  PLEASE UPDATE THE COLUMN NAMES ABOVE BASED ON YOUR DATASET\")\n",
    "print(\"Current configuration:\")\n",
    "print(f\"   Price Column: {price_col}\")\n",
    "print(f\"   Bedroom Column: {bedroom_col}\")\n",
    "print(f\"   Bathroom Column: {bathroom_col}\")\n",
    "print(f\"   Area Column: {area_col}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Check for missing values\n",
    "print(\"\\n🔍 Missing Values Analysis:\")\n",
    "missing_data = df_clean.isnull().sum()\n",
    "missing_percentage = (missing_data / len(df_clean)) * 100\n",
    "\n",
    "missing_df = pd.DataFrame({\n",
    "    'Missing Values': missing_data,\n",
    "    'Percentage': missing_percentage\n",
    "})\n",
    "missing_df[missing_df['Missing Values'] > 0].sort_values('Missing Values', ascending=False)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Handle missing values if any\n",
    "if missing_data.sum() > 0:\n",
    "    print(\"\\n🔄 Handling missing values...\")\n",
    "    # For numerical columns, fill with median\n",
    "    numeric_columns = df_clean.select_dtypes(include=[np.number]).columns\n",
    "    for col in numeric_columns:\n",
    "        df_clean[col].fillna(df_clean[col].median(), inplace=True)\n",
    "    print(\"✅ Missing values handled!\")\n",
    "else:\n",
    "    print(\"✅ No missing values found!\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Check for duplicates\n",
    "duplicates = df_clean.duplicated().sum()\n",
    "print(f\"\\n🔍 Number of duplicate rows: {duplicates}\")\n",
    "\n",
    "if duplicates > 0:\n",
    "    df_clean = df_clean.drop_duplicates()\n",
    "    print(f\"✅ Removed {duplicates} duplicate rows\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Univariate Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Distribution of Price\n",
    "if price_col in df_clean.columns:\n",
    "    fig, axes = plt.subplots(1, 3, figsize=(18, 5))\n",
    "    \n",
    "    # Histogram\n",
    "    axes[0].hist(df_clean[price_col], bins=50, edgecolor='black', alpha=0.7, color='skyblue')\n",
    "    axes[0].set_xlabel('Price')\n",
    "    axes[0].set_ylabel('Frequency')\n",
    "    axes[0].set_title('Distribution of House Prices')\n",
    "    \n",
    "    # Box plot\n",
    "    axes[1].boxplot(df_clean[price_col])\n",
    "    axes[1].set_ylabel('Price')\n",
    "    axes[1].set_title('Box Plot of House Prices')\n",
    "    \n",
    "    # KDE plot\n",
    "    df_clean[price_col].plot(kind='kde', ax=axes[2], color='red', linewidth=2)\n",
    "    axes[2].set_xlabel('Price')\n",
    "    axes[2].set_title('KDE of House Prices')\n",
    "    \n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "    \n",
    "    # Price statistics\n",
    "    print(\"\\n💰 Price Statistics:\")\n",
    "    print(df_clean[price_col].describe())\n",
    "else:\n",
    "    print(f\"❌ Price column '{price_col}' not found in dataset\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Distribution of Bedrooms\n",
    "if bedroom_col in df_clean.columns:\n",
    "    fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "    \n",
    "    # Count plot\n",
    "    bedroom_counts = df_clean[bedroom_col].value_counts().sort_index()\n",
    "    axes[0].bar(bedroom_counts.index, bedroom_counts.values, color='lightgreen', edgecolor='black')\n",
    "    axes[0].set_xlabel('Number of Bedrooms')\n",
    "    axes[0].set_ylabel('Count')\n",
    "    axes[0].set_title('Distribution of Bedrooms')\n",
    "    \n",
    "    # Pie chart\n",
    "    axes[1].pie(bedroom_counts.values, labels=bedroom_counts.index, autopct='%1.1f%%', startangle=90)\n",
    "    axes[1].set_title('Bedroom Distribution (%)')\n",
    "    \n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "    \n",
    "    print(\"\\n🛏️ Bedroom Statistics:\")\n",
    "    print(df_clean[bedroom_col].describe())\n",
    "else:\n",
    "    print(f\"❌ Bedroom column '{bedroom_col}' not found in dataset\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Distribution of Bathrooms\n",
    "if bathroom_col in df_clean.columns:\n",
    "    fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "    \n",
    "    # Count plot\n",
    "    bathroom_counts = df_clean[bathroom_col].value_counts().sort_index()\n",
    "    axes[0].bar(bathroom_counts.index, bathroom_counts.values, color='lightcoral', edgecolor='black')\n",
    "    axes[0].set_xlabel('Number of Bathrooms')\n",
    "    axes[0].set_ylabel('Count')\n",
    "    axes[0].set_title('Distribution of Bathrooms')\n",
    "    \n",
    "    # Pie chart\n",
    "    axes[1].pie(bathroom_counts.values, labels=bathroom_counts.index, autopct='%1.1f%%', startangle=90)\n",
    "    axes[1].set_title('Bathroom Distribution (%)')\n",
    "    \n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "    \n",
    "    print(\"\\n🚿 Bathroom Statistics:\")\n",
    "    print(df_clean[bathroom_col].describe())\n",
    "else:\n",
    "    print(f\"❌ Bathroom column '{bathroom_col}' not found in dataset\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Distribution of Area\n",
    "if area_col in df_clean.columns:\n",
    "    fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "    \n",
    "    # Histogram\n",
    "    axes[0].hist(df_clean[area_col], bins=50, edgecolor='black', alpha=0.7, color='orange')\n",
    "    axes[0].set_xlabel('Area (sq.ft.)')\n",
    "    axes[0].set_ylabel('Frequency')\n",
    "    axes[0].set_title('Distribution of Area')\n",
    "    \n",
    "    # Box plot\n",
    "    axes[1].boxplot(df_clean[area_col])\n",
    "    axes[1].set_ylabel('Area (sq.ft.)')\n",
    "    axes[1].set_title('Box Plot of Area')\n",
    "    \n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "    \n",
    "    print(\"\\n📐 Area Statistics:\")\n",
    "    print(df_clean[area_col].describe())\n",
    "else:\n",
    "    print(f\"❌ Area column '{area_col}' not found in dataset\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Bivariate Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create a subset with only the columns we need\n",
    "available_cols = []\n",
    "for col in [price_col, bedroom_col, bathroom_col, area_col]:\n",
    "    if col in df_clean.columns:\n",
    "        available_cols.append(col)\n",
    "\n",
    "if len(available_cols) >= 2:\n",
    "    df_subset = df_clean[available_cols].copy()\n",
    "    \n",
    "    # Rename columns for easier plotting\n",
    "    rename_dict = {}\n",
    "    if price_col in df_subset.columns:\n",
    "        rename_dict[price_col] = 'price'\n",
    "    if bedroom_col in df_subset.columns:\n",
    "        rename_dict[bedroom_col] = 'bedrooms'\n",
    "    if bathroom_col in df_subset.columns:\n",
    "        rename_dict[bathroom_col] = 'bathrooms'\n",
    "    if area_col in df_subset.columns:\n",
    "        rename_dict[area_col] = 'area'\n",
    "    \n",
    "    df_subset.rename(columns=rename_dict, inplace=True)\n",
    "    \n",
    "    print(\"✅ Created subset with columns:\", list(df_subset.columns))\n",
    "else:\n",
    "    print(\"❌ Not enough columns available for analysis\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Price vs Bedrooms\n",
    "if 'price' in df_subset.columns and 'bedrooms' in df_subset.columns:\n",
    "    plt.figure(figsize=(12, 6))\n",
    "    \n",
    "    # Box plot\n",
    "    plt.subplot(1, 2, 1)\n",
    "    df_subset.boxplot(column='price', by='bedrooms')\n",
    "    plt.title('Price vs Bedrooms')\n",
    "    plt.xlabel('Bedrooms')\n",
    "    plt.ylabel('Price')\n",
    "    \n",
    "    # Bar plot of average price by bedrooms\n",
    "    plt.subplot(1, 2, 2)\n",
    "    avg_price_by_bed = df_subset.groupby('bedrooms')['price'].mean()\n",
    "    avg_price_by_bed.plot(kind='bar', color='skyblue', edgecolor='black')\n",
    "    plt.title('Average Price by Bedrooms')\n",
    "    plt.xlabel('Bedrooms')\n",
    "    plt.ylabel('Average Price')\n",
    "    \n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "    \n",
    "    print(\"\\n💰 Average Price by Bedroom Count:\")\n",
    "    print(avg_price_by_bed)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Price vs Bathrooms\n",
    "if 'price' in df_subset.columns and 'bathrooms' in df_subset.columns:\n",
    "    plt.figure(figsize=(12, 6))\n",
    "    \n",
    "    # Box plot\n",
    "    plt.subplot(1, 2, 1)\n",
    "    df_subset.boxplot(column='price', by='bathrooms')\n",
    "    plt.title('Price vs Bathrooms')\n",
    "    plt.xlabel('Bathrooms')\n",
    "    plt.ylabel('Price')\n",
    "    \n",
    "    # Bar plot of average price by bathrooms\n",
    "    plt.subplot(1, 2, 2)\n",
    "    avg_price_by_bath = df_subset.groupby('bathrooms')['price'].mean()\n",
    "    avg_price_by_bath.plot(kind='bar', color='lightcoral', edgecolor='black')\n",
    "    plt.title('Average Price by Bathrooms')\n",
    "    plt.xlabel('Bathrooms')\n",
    "    plt.ylabel('Average Price')\n",
    "    \n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "    \n",
    "    print(\"\\n💰 Average Price by Bathroom Count:\")\n",
    "    print(avg_price_by_bath)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Price vs Area\n",
    "if 'price' in df_subset.columns and 'area' in df_subset.columns:\n",
    "    plt.figure(figsize=(14, 6))\n",
    "    \n",
    "    # Scatter plot\n",
    "    plt.subplot(1, 2, 1)\n",
    "    plt.scatter(df_subset['area'], df_subset['price'], alpha=0.5, c='green')\n",
    "    plt.xlabel('Area (sq.ft.)')\n",
    "    plt.ylabel('Price')\n",
    "    plt.title('Price vs Area')\n",
    "    \n",
    "    # Add trend line\n",
    "    z = np.polyfit(df_subset['area'], df_subset['price'], 1)\n",
    "    p = np.poly1d(z)\n",
    "    plt.plot(df_subset['area'], p(df_subset['area']), \"r--\", linewidth=2)\n",
    "    \n",
    "    # Hexbin plot for density\n",
    "    plt.subplot(1, 2, 2)\n",
    "    plt.hexbin(df_subset['area'], df_subset['price'], gridsize=30, cmap='YlOrRd')\n",
    "    plt.colorbar(label='Count')\n",
    "    plt.xlabel('Area (sq.ft.)')\n",
    "    plt.ylabel('Price')\n",
    "    plt.title('Price vs Area (Density Plot)')\n",
    "    \n",
    "    plt.tight_layout()\n",
    "    plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Correlation Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Correlation matrix\n",
    "if len(df_subset.columns) >= 2:\n",
    "    plt.figure(figsize=(10, 8))\n",
    "    \n",
    "    # Calculate correlation\n",
    "    corr_matrix = df_subset.corr()\n",
    "    \n",
    "    # Heatmap\n",
    "    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, \n",
    "                square=True, linewidths=1, cbar_kws={\"shrink\": 0.8})\n",
    "    plt.title('Correlation Matrix of Features', fontsize=16, fontweight='bold')\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "    \n",
    "    print(\"\\n📊 Correlation with Price:\")\n",
    "    if 'price' in corr_matrix.columns:\n",
    "        price_corr = corr_matrix['price'].drop('price')\n",
    "        for feature, corr in price_corr.sort_values(ascending=False).items():\n",
    "            strength = \"Strong\" if abs(corr) > 0.5 else \"Moderate\" if abs(corr) > 0.3 else \"Weak\"\n",
    "            print(f\"   {feature}: {corr:.3f} ({strength} correlation)\")\n",
    "else:\n",
    "    print(\"❌ Not enough columns for correlation analysis\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 9. Outlier Detection"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Function to detect outliers using IQR method\n",
    "def detect_outliers_iqr(data, column):\n",
    "    Q1 = data[column].quantile(0.25)\n",
    "    Q3 = data[column].quantile(0.75)\n",
    "    IQR = Q3 - Q1\n",
    "    lower_bound = Q1 - 1.5 * IQR\n",
    "    upper_bound = Q3 + 1.5 * IQR\n",
    "    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]\n",
    "    return outliers, lower_bound, upper_bound\n",
    "\n",
    "# Check for outliers in each column\n",
    "for col in df_subset.columns:\n",
    "    outliers, lower, upper = detect_outliers_iqr(df_subset, col)\n",
    "    outlier_count = len(outliers)\n",
    "    outlier_percent = (outlier_count / len(df_subset)) * 100\n",
    "    \n",
    "    print(f\"\\n📊 Outliers in '{col}':\")\n",
    "    print(f\"   Number of outliers: {outlier_count}\")\n",
    "    print(f\"   Percentage: {outlier_percent:.2f}%\")\n",
    "    print(f\"   Normal range: [{lower:.2f}, {upper:.2f}]\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize outliers using box plots\n",
    "fig, axes = plt.subplots(1, len(df_subset.columns), figsize=(15, 5))\n",
    "\n",
    "for i, col in enumerate(df_subset.columns):\n",
    "    axes[i].boxplot(df_subset[col])\n",
    "    axes[i].set_title(f'Box Plot - {col}')\n",
    "    axes[i].set_ylabel('Value')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 10. Feature Engineering"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create new features that might be useful\n",
    "df_features = df_subset.copy()\n",
    "\n",
    "# Price per square foot\n",
    "if 'price' in df_features.columns and 'area' in df_features.columns:\n",
    "    df_features['price_per_sqft'] = df_features['price'] / df_features['area']\n",
    "    print(\"✅ Created feature: price_per_sqft\")\n",
    "\n",
    "# Total rooms (bedrooms + bathrooms)\n",
    "if 'bedrooms' in df_features.columns and 'bathrooms' in df_features.columns:\n",
    "    df_features['total_rooms'] = df_features['bedrooms'] + df_features['bathrooms']\n",
    "    print(\"✅ Created feature: total_rooms\")\n",
    "\n",
    "# Bathroom to bedroom ratio\n",
    "if 'bathrooms' in df_features.columns and 'bedrooms' in df_features.columns:\n",
    "    df_features['bath_to_bed_ratio'] = df_features['bathrooms'] / df_features['bedrooms']\n",
    "    print(\"✅ Created feature: bath_to_bed_ratio\")\n",
    "\n",
    "# Area category\n",
    "if 'area' in df_features.columns:\n",
    "    df_features['area_category'] = pd.cut(df_features['area'], \n",
    "                                           bins=[0, 500, 1000, 1500, 2000, float('inf')],\n",
    "                                           labels=['Very Small', 'Small', 'Medium', 'Large', 'Very Large'])\n",
    "    print(\"✅ Created feature: area_category\")\n",
    "\n",
    "print(\"\\n📊 First few rows with new features:\")\n",
    "df_features.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 11. Model Preparation Insights"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"=\"*80)\n",
    "print(\"INSIGHTS FOR MODEL BUILDING\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "print(\"\\n🎯 Target Variable: Price\")\n",
    "print(f\"   - Distribution: {'Normal' if abs(df_features['price'].skew()) < 1 else 'Skewed'}\")\n",
    "print(f\"   - Skewness: {df_features['price'].skew():.2f}\")\n",
    "print(f\"   - Kurtosis: {df_features['price'].kurtosis():.2f}\")\n",
    "\n",
    "print(\"\\n🔑 Key Features and Their Relationships:\")\n",
    "print(\"   1. Area shows strong positive correlation with price\")\n",
    "print(\"   2. Bedrooms and bathrooms show moderate correlation\")\n",
    "print(\"   3. Consider log transformation if price is highly skewed\")\n",
    "\n",
    "print(\"\\n⚠️ Potential Issues:\")\n",
    "print(\"   1. Outliers present in all features - consider winsorization\")\n",
    "print(\"   2. Multicollinearity between bedrooms and bathrooms\")\n",
    "print(\"   3. Non-linear relationships might need polynomial features\")\n",
    "\n",
    "print(\"\\n📝 Recommended Features for Linear Regression:\")\n",
    "print(\"   - area (most important)\")\n",
    "print(\"   - bedrooms\")\n",
    "print(\"   - bathrooms\")\n",
    "print(\"   - Consider: price_per_sqft, total_rooms as additional features\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 12. Summary and Conclusions"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"=\"*80)\n",
    "print(\"SUMMARY AND CONCLUSIONS\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "print(\"\\n📌 Key Findings:\")\n",
    "print(\"   1. Dataset contains {} rows and {} columns\".format(df.shape[0], df.shape[1]))\n",
    "print(\"   2. Price range: {} to {}\".format(df_features['price'].min(), df_features['price'].max()))\n",
    "print(\"   3. Average house size: {:.0f} sq.ft.\".format(df_features['area'].mean()))\n",
    "print(\"   4. Most common configuration: {} BHK\".format(df_features['bedrooms'].mode()[0]))\n",
    "\n",
    "print(\"\\n📈 Model Recommendations:\")\n",
    "print(\"   1. Use Linear Regression as baseline model\")\n",
    "print(\"   2. Consider feature scaling (StandardScaler)\")\n",
    "print(\"   3. Split data: 80% training, 20% testing\")\n",
    "print(\"   4. Evaluate using R², RMSE, and MAE metrics\")\n",
    "\n",
    "print(\"\\n🎯 Next Steps:\")\n",
    "print(\"   1. Train linear regression model using train_model.py\")\n",
    "print(\"   2. Experiment with polynomial features\")\n",
    "print(\"   3. Try other algorithms (Random Forest, XGBoost)\")\n",
    "print(\"   4. Deploy the model using Flask web application\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 13. Save Cleaned Dataset"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Save the cleaned dataset for model training\n",
    "output_path = '../data/cleaned_housing_data.csv'\n",
    "df_features.to_csv(output_path, index=False)\n",
    "print(f\"✅ Cleaned dataset saved to: {output_path}\")\n",
    "print(f\"   Shape: {df_features.shape}\")\n",
    "print(f\"   Columns: {list(df_features.columns)}\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}